# 14 - Monthly event analysis

Estimate monthly event-timing models that separate the Matosinhos transition from the 2022 energy-stress period.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import pandas as pd

from portugal_refining_resilience.config import get_paths
from portugal_refining_resilience.config import load_analysis_config
from portugal_refining_resilience.events import EVENT_PHASES, fit_monthly_event_model, monthly_phase_summary
from portugal_refining_resilience.io import persist_dataframe

PATHS = get_paths(ROOT)
pd.set_option("display.max_columns", 100)


In [ ]:
panel_path = PATHS.processed / "fuel_monthly_analytical_panel.csv"
panel = pd.read_csv(panel_path, parse_dates=["date"])
# The monthly source runs 2002-2026, wider than the declared study window.
# Restrict it so the monthly models cover the same years as the annual work
# and the report, rather than quietly using data the paper does not claim.
_cfg = load_analysis_config(ROOT)
# JODI begins in 2002, later than the annual window, so the monthly arm
# declares its own start. Using the annual start would silently ask for
# twelve years the monthly source cannot supply.
_start = int(_cfg.get("monthly_start_year", _cfg["start_year"]))
_end = int(_cfg["end_year"])
panel = panel.loc[panel["date"].dt.year.between(_start, _end)].reset_index(drop=True)
print(f"monthly panel restricted to {_start}-{_end}: "
      f"{panel['date'].min():%Y-%m} .. {panel['date'].max():%Y-%m}, "
      f"{len(panel) // panel['product'].nunique()} months per product")
outcomes = [
    "imports_kt",
    "exports_kt",
    "demand_kt",
    "refinery_output_kt",
    "net_imports_kt",
    "net_import_to_demand_ratio",
]
available_outcomes = [column for column in outcomes if column in panel.columns]
panel[["date", "product", "event_phase", *available_outcomes]].head()


In [ ]:
summary_rows = []
for outcome in available_outcomes:
    part = monthly_phase_summary(panel, value_column=outcome)
    part.insert(1, "outcome", outcome)
    summary_rows.append(part)
phase_summary = pd.concat(summary_rows, ignore_index=True) if summary_rows else pd.DataFrame()
persist_dataframe(
    phase_summary,
    PATHS.metrics / "monthly_event_phase_summary.csv",
    key_columns=["product", "outcome", "event_phase"],
)
display(phase_summary)


In [ ]:
model_rows = []
for product, product_panel in panel.groupby("product"):
    for outcome in available_outcomes:
        model_frame = product_panel[["date", "month", "event_phase", outcome]].dropna()
        if len(model_frame) < 24:
            continue
        # The monthly panel opens in 2002 and the 2013 hydrocracker steps refinery
        # output and exports up inside it. Without a term for that step the pre-closure
        # trend every phase coefficient is measured against is fitted through it, which
        # is the misspecification the annual models were corrected for. Both are fitted
        # so the difference is visible rather than assumed away.
        for specification, control in (("no 2013 control", None), ("2013 held constant", 2013)):
            result = fit_monthly_event_model(
                model_frame, value_column=outcome, control_year=control
            )
            for term, estimate in result.params.items():
                model_rows.append(
                    {
                        "product": product,
                        "outcome": outcome,
                        "specification": specification,
                        "term": term,
                        "estimate": estimate,
                        "std_error": result.bse[term],
                        "p_value": result.pvalues[term],
                        "n_obs": int(result.nobs),
                        "r2_adj": result.rsquared_adj,
                    }
                )
monthly_models = pd.DataFrame(model_rows)
persist_dataframe(
    monthly_models,
    PATHS.metrics / "monthly_event_models.csv",
    key_columns=["product", "outcome", "specification", "term"],
)
# Show each phase with its slope companion: the level term is the shift at the phase
# boundary, not the whole effect, so the two are always read together.
phase_terms = [term for phase in EVENT_PHASES for term in (phase, f"{phase}_trend")]
display(monthly_models.loc[monthly_models["term"].isin(phase_terms)])


In [ ]:
plot_outcomes = [outcome for outcome in ["imports_kt", "exports_kt", "net_import_to_demand_ratio"] if outcome in available_outcomes]
for product, product_panel in panel.groupby("product"):
    for outcome in plot_outcomes:
        fig, ax = plt.subplots(figsize=(9, 4.8))
        product_panel.sort_values("date").plot(x="date", y=outcome, ax=ax, legend=False)
        for marker, label in [
            (pd.Timestamp("2021-05-01"), "Matosinhos transition"),
            (pd.Timestamp("2022-03-01"), "Energy stress"),
            (pd.Timestamp("2023-01-01"), "Post stress"),
        ]:
            ax.axvline(marker, color="0.35", linestyle="--", linewidth=1)
            ax.text(marker, ax.get_ylim()[1], label, rotation=90, va="top", ha="right", fontsize=8)
        ax.set_title(f"{product}: {outcome}")
        ax.set_xlabel("")
        ax.set_ylabel(outcome)
        fig.tight_layout()
        fig.savefig(PATHS.figures / f"monthly_event_{product}_{outcome}.png", dpi=160)
        plt.close(fig)
